# **Tutorial with Scripts and outputs for training a CNN for performing species delimitation using DeepID. We will use the *Euphorbia balsamifera* (sweet tabaiba) species complex.**

Manolo F. Perez; Ricarda Riina; Brant C. Faircloth; Marcelo B. Cioffi & Isabel Sanmartín. Integrative taxonomy using traits and genomic data for Species Delimitation with Deep learning.

### **Section 1: Defining the species delimitation hypotheses and performing simulations.**
The first step to use DeepID is to define explicit species delimitation hypotheses. DeepID is a Simulation Based Inference method, that uses a Supervised Deep Learning (with Convolution Neural Networks) to compare species delimitation hypotheses. For the *Euphorbia balsamifera* species complex, we compared 5 hypotheses, which can be visualized in the figure below.

<img src="img/Models_Emp.png" width="1000" height="500">

The simulations are carried on the population genetics simulator ms (https://home.uchicago.edu/~rhudson1/source/mksamples.html). To build the scenarios on the correct format for the software, we suggest looking at the examples in the ms manual (https://uchicago.app.box.com/s/l3e5uf13tikfjm7e1il1eujitlsjdx13/file/245765534272) and using PopPlanner (http://jjensenlab.org/wp-content/uploads/2016/02/PopPlanner.jar_.zip) to conceive the scenarios using a graphical interface.

#### **Section 1a: simulating data in ms.**

After conceiving the species delimitation scenarios, it is necessary to simulate data under those scenarios. We will use a python script to perform thousands of simulations for each scenario using different values (sampled from prior distributions) for the parameters related to the speciation process.

For the *Euphorbia balsamifera* species complex, we will use the [´simulate_ms_Euphorbia.py´](https://github.com/manolofperez/DeepID/blob/master/EmpiricalData/Euphorbia/simulate_ms_Euphorbia.py) script.

The main changes that are necessary, in case you want to apply it to a different species, would be related to modifying the number of samples, the prior values for theta (mutation rate and population size) and the ms command used to simulate the scenarios specific to your focal species. These potential changes are detailed below.

The script starts importing the required libraries and defining functions to read ms simulations and genealogies in the correct format, as well as auxiliary functions to sample divergence times for intra and interspecific nodes. There is no need to change any of the code here (reproduced below for clarity).

```python

#!/usr/bin/python3

## in order to use this code you have to have ms installed on your computer
## ms can be freely downloaded from:
## http://home.uchicago.edu/rhudson1/source/mksamples.html

#Import required libraries
import random
import os
import math
import shlex, subprocess
import numpy as np

##define a function to read ms' simulations and transform them into a NumPy array.    
def ms2nparray(xfile):
	g = list(xfile)
	k = [idx for idx,i in enumerate(g) if len(i) > 0 and i.startswith(b'//')]
	f = []
	for i in k:
		L = g[i+5:i+N_allpops+5]
		q = []
		for i in L:
			i = [int(j) for j in list(i.decode('utf-8'))]
			i = np.array(i, dtype=np.int8)
			q.append(i)
		q = np.array(q)
		q = q.astype("int8")
		f.append(np.array(q))
	return f


##define a function to read ms' coalescent trees simulations and add them to a list.
def get_newick(xfile):
	g = list(xfile)
	k = [idx for idx,i in enumerate(g) if len(i) > 0 and i.startswith(b'//')]
	t = []
	for i in k:
		n = g[i+1]
		t.append(n.decode('utf-8'))
	return t


def get_interspecific_tau(theta_a, theta_b, p_min=0.5, p_max=1.0, T_anc=None):
    """
    Ensures both populations are above the p_min threshold.
    The larger population (max theta) is the limiting factor.
    """
    limiting_theta = max(theta_a, theta_b)
    # We use a loop to ensure T is younger than T_parent.
    while True:
        p_target = np.random.uniform(p_min, p_max)
        # Calculate tau based on the lineage that takes longer to sort.
        ## obtain divergence time (tau - τ) priors using the P values. e.g., Pi_A = 1−2/3*e^(−2τ/θ_A); τ = -(θ_A*ln(-(3*Pi_A-3)/2)/2).
        tau = -((limiting_theta)*math.log(-(3*p_target-3)/2)/2)
        ## Transform divergence times to 4Ne generations units (required by ms).
        T = 4*tau/limiting_theta
        # If there's no parent, or if T is younger than parent return calculated values.
        if T_anc is None or T < T_anc:
            return T, p_target

def get_intraspecific_tau(theta_a, theta_b, p_min=1/3, p_max=0.4):
    """
    Ensures both populations are below the p_max threshold.
    The smaller population (min theta) is the limiting factor.
    """
    limiting_theta = min(theta_a, theta_b)
    p_target = np.random.uniform(p_min, p_max)
    
    # Calculate tau based on the lineage that sorts faster. 	
    ## obtain divergence time (tau - τ) priors using the P values. e.g., Pi_A = 1−2/3*e^(−2τ/θ_A); τ = -(θ_A*ln(-(3*Pi_A-3)/2)/2).
    tau = -((limiting_theta)*math.log(-(3*p_target-3)/2)/2)
    ## Transform divergence times to 4Ne generations units (required by ms).
    T = 4*tau/limiting_theta
    return abs(T),p_target

```

Next, we have to declare some important variables for the simulations. We have to define the number of simulations per model (10K per model is a good start), as well as the sample sizes for each population (species), which one would have to. hange to acommodate the specific empirical example being studied. In the  *E. balsamifera* species complex we have samples divided in three putative species (*E. balsamifera*, *E. adenensis* and *E. sepium*). Note here that the number of samples is actually the number of chromosomes sampled, so if you have diploid species with genomic information available for the two chromosomes, you have to provide the number of sampled individuals multiplied by two.

```python
### variable declarations

#define the number of simulations
Priorsize = 10000

## sample size of Euphorbia balsamifera subsp. adenensis.
N_adenensis = 19
## sample size of Euphorbia balsamifera subsp. balsamifera.
N_balsamifera = 80
## sample size of Euphorbia balsamifera subsp. sepium.
N_sepium = 10

## sample size for all pops combined.
N_allpops = N_adenensis + N_balsamifera + N_sepium
```

The next part is to create files (for sampled parameter values) and variables to store the results (for the genomic data and the trees). There should be one per scenario (5 scenarios in the case of the *E. balsamifera* species complex) and you can use informative names for each scenario (e.g., 3sp_M for the scenario with 3 species and higher migration rates).

```python
## create files to store parameters, trees and the models
os.mkdir("trainingSims")
par_1sp = open("par_1sp.txt","w")
par_2spMorph = open("par_2spMorph.txt","w")
par_2spPhylo = open("par_2spPhylo.txt","w")
par_3sp = open("par_3sp.txt","w")
par_3sp_M = open("par_3sp_M.txt","w")

## create lists to store trees from each scenario
trees_1sp = []
trees_2spMorph = []
trees_2spPhylo = []
trees_3sp = []
trees_3sp_M = []

## create lists to store simulations from each scenario
Model_1sp = []
Model_2spMorph = []
Model_2spPhylo = []
Model_3sp = []
Model_3sp_M = []
```


We now have to define the loops where simulatation will be performed (with a "Priorsize" number of times per scenario). For example, the first scenario being simulated in the  *E. balsamifera* species complex consists in all populations (putative species) being merged in a single species. At the beggining of the loop we define a baseline $\theta$ (per site per generation). In this example we used an estimate of $\theta$ obtained from sequence data and divided the obtained value by the sequence size (1700 bp). This baseline $\theta$ will be different for other study cases, so the user should change this accordingly.

```python
####Simulate the species delimitation scenarios####

### Scenario 1: All populations belong to one species
for i in range(Priorsize):
	### Define parameters
    ## Theta (4Neu) values between 2 and 5 according to Rincon-Barrado et al (2024), divided by the size of the alignments (1700 bp)
	Theta = random.uniform(2,5)/1700
```

We also allow for the specific $\theta$ of each population (putative species) to vary, so we sample a relative size multiplicator for each population. In this example specific population sizes are sampled from a uniform distribution with values that allow their size to be anywhere from half to double of the baseline population size.

```python
	#Now sample relative Thetas (pop sizes) for every deme.
	Theta_A, Theta_B, Theta_S = np.random.uniform(0.5, 2, size=3)
```

After defining the population sizes, we now sample migration rates from uniform distributions. The range of values sampled will depend on whether we are considering the populations as belonging to the same species or not. Following Rannala and Yang (2022), we assume intraspecific migration rates to be between Nem = 1 and 5 (very high migration). In ms, we can provide a matrix with migration rates among the samples.

```python
	#And sample migration rates between Nm = 1 and 5 for all of the demes.
	pops = ['A','B','S']
	n = len(pops)

	# draw every possible rate (including i→i) between Nm = 1 and 5 (ms requires migration rates as 4Nm, so bounds are 4 and 20)
	M = np.random.uniform(4, 20, size=(n, n))
	# zero out self-migration
	np.fill_diagonal(M, 0)	
```

Finally, we sample $\tau$ and the limiting value of either Pa and Pb for each node. In this model all nodes are intraspecific splittings, so we use the function for obtaining intraspecific values of tau.

```python
	## Sample Pi values for the deepest node (Pi1_A and Pi1_S) between 1/3 (minimum value -> 0.333) and 0.4 (same species).
	T1, P1 = get_intraspecific_tau(Theta*Theta_A, Theta*Theta_S)

	## Sample Pi values for the second deepest node (Pi2) with a smaller value than deepest one (Pi1).
	T2 = np.random.uniform(0, T1)
```

The last part of the loop is dedicated to translate all sampled parameters from the previous step into a ms command that simulates genomic data. The ms command will be specific to the scenario being simulated, the sampled values of the simulation being carried and the sample sizes of the empirical dataset being modelled. Therefore, this command needs to be completely changed for other empirical cases. As stated above, there are some software that can help conceiving and visualizing the ms commands to make sure the scenario is being properly modelled.

```python
	## ms command
	com=subprocess.Popen("./ms %d 428 -s 1 -t %f -I 3 %d %d %d -n 1 %f -n 2 %f -n 3 %f -ma %s -ej %f 2 1 -ej %f 1 3 -T" % (N_allpops, Theta, N_adenensis, N_balsamifera, N_sepium, Theta_A, Theta_B, Theta_S, ' '.join(map(str, M.flatten())), T2, T1), shell=True, stdout=subprocess.PIPE).stdout
```

After the ms command, the script saves the resulting simulated data in the correct format, using the variables and the files defined at the beggining of the script.

```python
	output = com.read().splitlines()
	Model_1sp.append(np.array(ms2nparray(output)).swapaxes(0,1).reshape(N_allpops,-1).T)

	## save parameter values and trees
	par_1sp.write("%f\t%f\t%f\t%f\t%f\t%f\n" % (Theta,Theta_A, Theta_B, Theta_S,T1,T2))
```
Note that in the line where you are sampling the genealogies from the simulated data, you need to change the value (4 in the example) to match the number of traints being simulated.

```python
	#Randomly save a number of tree equivalent to the number of traits
	trees_1sp.append(random.sample(get_newick(output),4))
	print("Completed %d %% of Model 1 simulations" % (float(i)/Priorsize*100))
```

The same type of change needs to be done for the other scenario. To illustrate a more complex example, the scenario with 3 species and higher migration have more instances where one would have to change.
In this case, the migration matrices are more complex, as the three putative species are actual species, so we need to model both interspecific gene flow, and these matrices change through time. So we first define the migration matrix for the present. The insterspecific migration rate is Nem between 0 and 0.1.

```python
	species = [
    [0],  # A
    [1],  # B
    [2],  # S
	]
	
	#Now change it for different species (A,B,S) with a value of m (Nm) = 0.1; 4Nm = 0.4
	for sp in species:
		for i in sp:
			for j in range(n):
				if j not in sp:
					M[i, j] = np.random.uniform(0, 0.4)
					M[j, i] = np.random.uniform(0, 0.4)
```

Then we need to create matrices with higher migration rates (higher than intraspecific, but not so high as interspecific). We sample those with Nem between 0.1 and 0.5. Also, we have two splits, so we need to sample two matrices for the two periods where migration is higher among incipient species.

```python
	# Now create a migration matrix with higher rates between species after the divergence (reduced after a period of time - up to 1/2 of the divergence time between species.)
	# draw every possible rate (including i→i) between Nm = 0.1 to 0.5 (ms requires migration rates as 4Nm, so bounds are 0.4 and 2)
	
	M_div_2 = np.array(M)

	#Now change it for (A,S) with a value of m (Nm) from 0.1 to 0.5; 4Nm = [0.4,2]
	for i in [0]:
		for j in [1]:
			M_div_2[i, j] = np.random.uniform(0.4,2)
			M_div_2[j, i] = np.random.uniform(0.4,2)

	# Now create a migration matrix with higher rates betwee species after the divergence (reduced after a period of time - up to 1/2 of the divergence time between species.)
	
	# draw every possible rate (including i→i) between Nm = 1 and 5 (ms requires migration rates as 4Nm, so bounds are 4 and 20)
	M_div_1 = np.array(M_div_2)	

	#Now change it for (A,B) with a value of m (Nm) from 0.1 to 0.5; 4Nm = [0.4,2]
	for i in [0]:
		for j in [2]:
			M_div_1[i, j] = np.random.uniform(0.4,2)
			M_div_1[j, i] = np.random.uniform(0.4,2)
```

Apart from those vhanges in the migration rates, we now have two interspecific splitting nodes, so we need to use the respective function to sample their $\tau$ values.

```python
	## Sample Pi values for the deepest node (Pi1_A and Pi1_S) between 0.5 and 1 (different species).
	T1, P1 = get_interspecific_tau(Theta*Theta_A, Theta*Theta_S)

	## Sample Pi values for the second node (Pi1_A and Pi1_B) higher than 0.5 (different species).
	T2, P2 = get_interspecific_tau(Theta*Theta_A, Theta*Theta_B, p_max=P1, T_anc=T1)
```

The last change is to sample the time periods for when the migration rates will be higher.

```python
	## Sample Pi values for the deepest node (Pi1_A and Pi1_S) between 0.5 and 1 (different species).
	T1, P1 = get_interspecific_tau(Theta*Theta_A, Theta*Theta_S)

	## Sample Pi values for the second node (Pi1_A and Pi1_B) higher than 0.5 (different species).
	T2, P2 = get_interspecific_tau(Theta*Theta_A, Theta*Theta_B, p_max=P1, T_anc=T1)
```

#### **Section 1b: simulating traits in R.**

After running the whole simulation script, we obtain a the genomic data (.npz files), the genealogies (trees.npz file) and the parameters (.txt files) for each scenario. The genomic data is ready for the training procedure, but we still need to simulate the traits. For that, we will use the [SimulateTraits.R](https://github.com/manolofperez/DeepID/blob/master/SimulatedData/SimpleSpeciationScenarios/SimulateTraits.R) script.

Again, the first par of the script is related to loading libraries and defining general variables, so there is no need to change anything here.

```R
# A script to simulate continuous traits on several newick trees saved in a file.

# To install the required packages, uncomment the lines below
# install.packages("ape",repos="https://cloud.r-project.org")
# install.packages("geiger",repos="https://cloud.r-project.org")
# install.packages("phytools",repos="https://cloud.r-project.org")
# install.packages("reticulate")
# install.packages(c( "foreach", "doParallel") )

# Load the required packages.
library(ape)
library(geiger)
library(phytools)
library(foreach)
library(doParallel)
library(reticulate)
# import the NumPy function from the package reticulate (interface with Python)
np <- import("numpy",convert=F)

#setup parallel backend to use many processors.
cl <- makeCluster(5) #This is to use 10 cores, adjust accordingly (leaving at least one free core). Alternatively one can use detectCores() function.
registerDoParallel(cl)

#Create a folder for storing the simulated traits.
dir.create("traits")
```

Next, we need to define the number of traits that will be simulated. This needs to be changed, as each empirical data set will have its own features.
For the empirical example here, we have 4 continuous traits, so we need to change this to 4.

```R
# Number of total traits for each simulation.
#nsims <- 100 # the original script was simulating 100 traits, so I commented this line
nsims <- 4 # the new script will have 4 traits here
# Read the total number of trees (here it was 100 trees for each simulated data set).
ntrees <- np$load("trees.npz")

# Load trees as a matrix, with the number of rows equal to the number of simulated datasets and 100 columns (trees per simulated data set).
#trees<-matrix(ntrees$f[["trees"]], ncol=100,byrow=TRUE) # same here, I commented the original line, as it was simulating 100 traits
trees<-matrix(ntrees$f[["trees"]], ncol=4,byrow=TRUE) # the new script has 4 traits.
```


Now the script will loop through the simulated trees and simulate traits along those genealogies. For example, if you are simulating a trait under the Brownian Motion (BM) model, the loop looks like this:

```R
### Simulate traits ###
# BM
traits_BM={}

# start a parallelized loop that reads the trees in the matrix and assigns coordinates (i,j) to each tree.
traits_BM<-foreach(i=1:nrow(trees),.combine=rbind,.packages = c("phytools","ape","geiger")) %:%
  foreach(j=1:nsims,.combine=cbind,.packages = c("phytools","ape","geiger")) %dopar% {
    # read the current tree.
    tree <- read.tree(text=trees[i,j])
    # each diploid individual has two tips in the tree, so we need to drop one of them (the second) to correctly simulate the traits.
    tree <-drop.tip(tree, as.character(seq(0,length(tree$tip.label),by=2)))
    # simulate traits using the topology.
    sims <- fastBM(tree, a = 0, sig2 = 0.06, nsim = 1) 		
    #order the traits individuals according to the SNPs order.
    ordered_sims <- sims[order(as.numeric(names(sims)))]
    return(ordered_sims)

# save the output.
write.table(format(as.matrix(traits_BM), width = 6),"./traits/traits_BM.txt",row.names=F,col.names=F,quote = F)
```

The only changes you would have to do would be in the "fastBM" command, in case you want to change the parameters of this model to simulate the traits. Otherwise you can use the defaults here. The script will repeat this procedure for the OU and the discrete traits (if applicable).

### **Section 2: Training DeepID**

After simulating the traits, you have everything you need to train the neural network. Below we present the code from the Jupyter nodebook for training DeepID on the *E. balsamifera* species complex ([Train_Test_Predict_Euphorbia.ipynb](https://github.com/manolofperez/DeepID/blob/master/EmpiricalData/Euphorbia/Train_Test_Predict_Euphorbia.ipynb)).

#### **Section 2a: Building the CNN.**

The notebook starts by importing all libraries required for the training procedure.

In [ ]:
# import all required libraries
import sys, os
import numpy as np
import pandas as pd
import random
from random import shuffle, choice
import time
import os
import glob
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers
from random import shuffle, choice
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler,StandardScaler

Next, we define the CNN architecture and the gated concatenation layer. There is no need to change these functions and the text is annotated to help the user change it in case needed (not recommended).

In [ ]:
# define a function to build a CNN for the SNP data.
def create_cnn(xtest, regularizer=None):
  # obtain the input dimensions.
  inputShape = (xtest.shape[1], xtest.shape[2])
  inputs = Input(shape=inputShape)
  x = inputs
  # first convolutional layer, remember to remove bias if you are intercalating with batch normalization.
  x = Conv1D(256, kernel_size=3, activation='relu', use_bias=False)(x)
  # batch normalization.
  x = BatchNormalization()(x)
  # second layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # third layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # pool the CNN outputs.
  x = GlobalMaxPooling1D()(x)
  # this part is similar to the MLP, a fully connected neural network. We intercalated with dropout to reduce overfitting.
  x = Dense(128, activation='relu')(x)
  # dropout.
  x = Dropout(0.5)(x)
  # second layer of the fully connected neural network.
  x = Dense(128, activation='relu')(x)
  x = Dropout(0.5)(x)
  # third layer of the fully connected neural network. This one matches the number of nodes coming out of the MLP.
  x = Dense(64, activation='relu')(x)
  # Construct the CNN
  #x = BatchNormalization()(x)#Not working very well
  #x = LayerNormalization()(x)#Better?
  model = Model(inputs, x)
  # Return the CNN
  return model

class GatedConcatenate(Layer):
    """
    Applies a trainable or fixed gate (weight) to each input branch
    before concatenating them.
    
    Args:
        initial_traits_weight (float): The starting weight for the first input (traits).
                                     Must be between 0 and 1. The weight for the
                                     second input (SNPs) will be (1 - this value).
        trainable_gates (bool): If True, the model can learn to adjust these
                                weights. If False, the weights are fixed.
    """
    def __init__(self, initial_traits_weight, trainable_gates=True, **kwargs):
        super(GatedConcatenate, self).__init__(**kwargs)
        if not (0 <= initial_traits_weight <= 1):
            raise ValueError("initial_traits_weight must be between 0 and 1.")
            
        self.initial_weights = [initial_traits_weight, 1.0 - initial_traits_weight]
        self.trainable_gates = trainable_gates

    def build(self, input_shape):
        # Create the gate variables. They are shaped for broadcasting across the features.
        self.gates = self.add_weight(
            name='gates',
            shape=(1, len(input_shape)), # Shape will be (1, 2)
            initializer=tf.constant_initializer(self.initial_weights),
            trainable=self.trainable_gates,
            dtype=tf.float32
        )
        super(GatedConcatenate, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("GatedConcatenate expects a list of exactly two input tensors.")
        
        # Apply the gates (weights) to each branch using element-wise multiplication
        gated_traits = inputs[0] * self.gates[0, 0]
        gated_snps = inputs[1] * self.gates[0, 1]
        
        # Concatenate the scaled branches
        return Concatenate()([gated_traits, gated_snps])

    def get_config(self):
        # Needed for saving/loading the model
        config = super().get_config()
        config.update({
            'initial_traits_weight': self.initial_weights[0],
            'trainable_gates': self.trainable_gates,
        })
        return config
    
def gated_contributions(model, layer_name=None, labels=("traits", "SNPs")):
    # 1) find the layer
    gated_layer = model.get_layer('gated_concatenate')
    weights = gated_layer.get_weights()[0][0]
    rel_weight = np.sum(np.abs(weights))
    print(f"Final learned weights: Traits={weights[0]/rel_weight:.4f}, SNPs={weights[1]/rel_weight:.4f}")

We also define variables (hyperparameters) that will be used during training. The only hyperparameter that needs to be changed is "num_classes", as this is the number of scenarios being compared. In this example we have 5 scenarios.

In [ ]:
## define variables that will be used to train all networks.
# size of the minibatches containing simulations are passed through the network in each epoch.
batch_size = 256
# number of training iterations (epochs) for the SNP only and the combined networks.
epochs = 100
# number of training iterations (epochs) for the traits only networks.
num_classes = 5

#### **Section 2b: Train the network with 10,000 simulations from each model**

Now we are going to load the simulated data and train the network. First, we load the traits and standard scale them to avoid bias coming from traits measured with different scales. 

In [ ]:
traits_BM = []
#c_traits = np.stack([np.loadtxt(f) for f in all_files])
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(50000,-1,4)

traits_BM = np.array(traits_BM)

#Use standard scaling for the continuous traits.
scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)


traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(50000,-1,4)

traits_OU = np.array(traits_OU)

#Use standard scaling for the continuous traits
scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

We have to add missing individuals in *E. balsamifera*, as we have more samples with SNPs than with traits in this putative species.

In [ ]:
#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_BM.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_BM[i,j,:] = 0
    
#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_OU.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_OU[i,j,:] = 0

Then we load the simulated SNPs as 3-dimensional NumPy arrays (simulation, SNPs, samples). We transform major alleles in -1, minor alleles in 1 and missing data (randomly scattered through the matrix) in 0s. We also associate each simulation with the appropriate label (y).

In [ ]:
u1 = np.load("./trainingSims/Model_1sp.npz")
u2 = np.load("./trainingSims/Model_2spMorph.npz")
u3 = np.load("./trainingSims/Model_2spPhylo.npz")
u4 = np.load("./trainingSims/Model_3sp.npz")
u5 = np.load("./trainingSims/Model_3sp_M.npz")

#change to Model_3sp_M
x=np.concatenate((u1['Model_1sp'],u2['Model_2spMorph'],u3['Model_2spPhylo'],u4['Model_3sp'],u5['Model_3sp_M']),axis=0)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(x):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      x[arr][idx][x[arr][idx] == 1] = -1
      x[arr][idx][x[arr][idx] == 0] = 1
    else:
      x[arr][idx][x[arr][idx] == 0] = -1

y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2spMorph']))])
y.extend([2 for i in range(len(u3['Model_2spPhylo']))])
y.extend([3 for i in range(len(u4['Model_3sp']))])
y.extend([4 for i in range(len(u5['Model_3sp_M']))])
y = np.array(y)

#Add missing data as 0s, according to a specifies missing data percentage
missD_perc = 17.7
missD = int(x.shape[1]*x.shape[2]*(missD_perc/100))
for i in range(x.shape[0]):
  for m in range(missD):
    j = random.randint(0, x.shape[1] - 1)
    k = random.randint(0, x.shape[2] - 1)
    x[i][j][k] = 0

del(missD)

Now we define a function to load the data, transform it into the correct format and feed it to the network.

```python
################################################################################################################################################
# We will start with traits simulated under the BM model.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and BM traits only).

# function to train on the combined datasets
def combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_BM_train)
    snps = create_cnn(xtrain)
```

We also define the gated concatenation layer. We start with an 50/50 contribution for each branch (SNPs and traits), but let the model learn. It is also possible to set pre-defined weights for each branch, by changing the "initial_traits_weight" to a different traits relative contribution (from 0 to 1). To keep the weight of each branch fixed (meaning the network will not try to learn the best weights), change "trainable_gates" to false.

```python
    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])
```

The reamaining of the code here is just to define the architecture of the final layers, set the hyperparameters related to optimizers, loss function and early stopping strategy and fit the model (exporting the contributions of each branch on the gated concatenation). There is no need to change anything here.

```python
    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_BM_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_BM_test, xtest], ytest),callbacks=[earlyStopping])

    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')

    return model
```

We also define functions to perform the same procedure, but training the network only with SNPs or only with traits. Again, no need to change anything here.

In [ ]:
# function to train on the SNP only datasets
def SNP_subset(ytrain, ytest, xtrain, xtest):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]

    # Create the the CNN 
    snps = create_cnn(xtest)
    
    #Create the last layer for the SNP network
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(xtrain, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(xtest, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    trait = create_cnn(traits_BM_train)
    
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_BM_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_BM_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

Now we per form the actual training. We need to separate the training simulations into the training and validation sets. We use a 75/25 split and we shuffle the simulations.

In [ ]:
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,x,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model1 = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

model1.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')

We also train wuth the SNP-only data.

In [ ]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,x,test_size=0.25, shuffle=True,stratify=y)

# train the network
model2 = SNP_subset(ytrain, ytest, xtrain, xtest)

model2.save(filepath='./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')

And with the trait-only data.

In [ ]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model3 = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)
          
model3.save(filepath='./TrainedModels/Trained_Traits_Model_BM.acc.mod')

We repeat the same procedure with the Ornstein-Uhlenbeck (OU) model.

In [ ]:
# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and OU traits only).

# function to train on the combined datasets
def combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_OU_train)
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_OU_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_OU_test, xtest], ytest),callbacks=[earlyStopping])
    
    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the OU trait only datasets
def OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    trait = create_cnn(traits_OU_train)

    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_OU_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_OU_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

We train the combined network and the network for traits-only.

In [ ]:
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,x,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model4 = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

model4.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')

In [ ]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model5 = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)
          
model5.save(filepath='./TrainedModels/Trained_Traits_Model_OU.acc.mod')

### **Section 3: Perform cross validation predictions with another 1000 simulations per model. Those were not seen by the network during training**

Now we are going to test the trained model using the test set. These are new simulations that were not used by the network during the training. The idea here is to use simulations as if they are our real data (we call this pseudoobserved data in ABC) and see what the network predicts for each of these simulations. After that we compare the predicted values with the real (simulated) ones.

First we load the test data set, on the same way we did above for the trining data set.

In [ ]:
model1 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')
model2 = load_model('./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')
model3 = load_model('./TrainedModels/Trained_Traits_Model_BM.acc.mod')
model4 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')
model5 = load_model('./TrainedModels/Trained_Traits_Model_OU.acc.mod')

traits_BM = []
traits_BM = np.loadtxt("./testSims/traits/traits_BM.txt").reshape(5000,-1,4)

traits_BM = np.array(traits_BM)
Traits_BM = np.array(traits_BM)

#Use standard scaling for the continuous traits.
for i in range(traits_BM.shape[2]):
    traits_BM[:, :, i] = scalers_BM[i].transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)


traits_OU = []
traits_OU = np.loadtxt("./testSims/traits/traits_OU.txt").reshape(5000,-1,4)

traits_OU = np.array(traits_OU)

#Use standard scaling for the continuous traits.
for i in range(traits_OU.shape[2]):
    traits_OU[:, :, i] = scalers_OU[i].transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_BM.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_BM[i,j,:] = 0
    
#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_OU.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_OU[i,j,:] = 0

u1 = np.load("./testSims/Model_1sp.npz")
u2 = np.load("./testSims/Model_2spMorph.npz")
u3 = np.load("./testSims/Model_2spPhylo.npz")
u4 = np.load("./testSims/Model_3sp.npz")
u5 = np.load("./testSims/Model_3sp_M.npz")

xtest=np.concatenate((u1['Model_1sp'],u2['Model_2spMorph'],u3['Model_2spPhylo'],u4['Model_3sp'],u5['Model_3sp_M']),axis=0)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(xtest):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      xtest[arr][idx][xtest[arr][idx] == 1] = -1
      xtest[arr][idx][xtest[arr][idx] == 0] = 1
    else:
      xtest[arr][idx][xtest[arr][idx] == 0] = -1

#Add missing data as 0s, according to a specifies missing data percentage
#46,761 SNP datapoints and 8,287 missing genotypes = 17.7%
missD_perc = 17.7
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    j = random.randint(0, xtest.shape[1] - 1)
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i][j][k] = 0


ytest=[0 for i in range(len(u1['Model_1sp']))]
ytest.extend([1 for i in range(len(u2['Model_2spMorph']))])
ytest.extend([2 for i in range(len(u3['Model_2spPhylo']))])
ytest.extend([3 for i in range(len(u4['Model_3sp']))])
ytest.extend([4 for i in range(len(u5['Model_3sp_M']))])
ytest = np.array(ytest)

Then, we make predictions and export them as confusion matrices.

In [ ]:
#first get the predictions
pred = model1.predict([np.swapaxes(traits_BM,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model2.predict(xtest)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model3.predict(np.swapaxes(traits_BM,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model4.predict([np.swapaxes(traits_OU,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model5.predict(np.swapaxes(traits_OU,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

### **Section 4: Predict the most likely model for the empirical data, using the trained CNN.**

We now load the empirical data, transform it in thesame way we did above for the trained data and make predictions with the trained network.

In [ ]:
# Load the SNP data set
inSNPs=np.loadtxt("./input_SNPs.txt")
EmpSNPs = np.array(inSNPs)

#transform major alleles in -1 and minor 1
for idx,row in enumerate(EmpSNPs):
  if np.count_nonzero(row==1) > np.count_nonzero(row==-1):
    EmpSNPs[idx][EmpSNPs[idx] == 1] = 9
    EmpSNPs[idx][EmpSNPs[idx] == -1] = 1
    EmpSNPs[idx][EmpSNPs[idx] == 9] = -1

# We create 100 datasets with the repeated SNPs data. We will combine this with the resampled trait data to create 100 bootstrap replicates.
EmpSNPs = np.repeat(EmpSNPs[np.newaxis, :, :], 100, axis=0)

#load the trait data
ade=np.genfromtxt("./input_traits_ade.txt", delimiter="\t", filling_values=0)
bal=np.genfromtxt("./input_traits_bal.txt", delimiter="\t", filling_values=0)
sep=np.genfromtxt("./input_traits_sep.txt", delimiter="\t", filling_values=0)

ade=np.array(ade)
bal=np.array(bal)
sep=np.array(sep)

#resample the trait data to build 100 bootstrap replicates
res = []
for i in range(0,100):
  idx_ade = np.random.choice(ade.shape[0], 19, replace=False)
  n = ade[idx_ade,:]
  idx_bal = np.random.choice(bal.shape[0], 29, replace=False)
  n_z = np.zeros((80,4))
  for i in range(len(idx_bal)):
    idx_nz=np.random.choice(80, len(idx_bal), replace=False)
    n_z[idx_nz[i]] = bal[idx_bal[i],:]
  n = np.concatenate((n,n_z), axis=0)
  idx_sep = np.random.choice(sep.shape[0], 10, replace=False)
  n = np.concatenate((n,sep[idx_sep,:]), axis=0)
  res.append(np.array(n))

traits = np.array(res)

emp_traits = np.array(traits)

#Use standard scaling for the continuous traits.
for i in range(emp_traits.shape[2]):
    emp_traits[:, :, i] = scalers_BM[i].transform(emp_traits[:, :, i]) 

Now we make predictions for each of the trained networks. Those are the final prediction, which we use to define the softmax probability of each scenario.

In [ ]:
Emp_comb_BM_pred = model1.predict([np.swapaxes(emp_traits,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_BM_Predictions.txt", Emp_comb_BM_pred)

Emp_SNP_pred = model2.predict(EmpSNPs)

np.savetxt("Pred_Emp_SNP_Predictions.txt", Emp_SNP_pred)

Emp_BM_traits_pred = model3.predict(np.swapaxes(emp_traits,1,2))

np.savetxt("Pred_Emp_traits_BM_Predictions.txt", Emp_BM_traits_pred)

emp_traits = np.array(traits)

#Use standard scaling for the continuous traits.
for i in range(emp_traits.shape[2]):
    emp_traits[:, :, i] = scalers_OU[i].transform(emp_traits[:, :, i]) 
    
Emp_comb_OU_pred = model4.predict([np.swapaxes(emp_traits,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_OU_Predictions.txt", Emp_comb_OU_pred)

Emp_OU_traits_pred = model5.predict(np.swapaxes(emp_traits,1,2))

np.savetxt("Pred_Emp_traits_OU_Predictions.txt", Emp_OU_traits_pred)